In [71]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import AutoTokenizer, AutoModelForMaskedLM

from transformers import AutoModelForCausalLM
from tokenizers import Tokenizer
import torch
import torch.nn.functional as F

import einops

In [10]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)  

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

In [3]:
device = "cuda"

In [29]:
# 35 M
model_name = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

<img src="./ESM2_architecture.png" width="1000"/>

In [80]:
hooked_esm_config = HookedTransformerConfig(
    n_layers=12,
    d_model=480,
    d_head=24, # 480 -> 480 = 24 * 20
    n_heads=20,
    d_mlp=1920,
    d_vocab=33,
    n_ctx=59,
    act_fn="gelu",
    normalization_type="LN",
    positional_embedding_type="rotary",
    attention_dir="bidirectional"
)

In [81]:
test = HookedTransformer(hooked_esm_config)

In [106]:
def get_hooked_state_dict(hf_esm_state_dict, cfg):
    """
    hugging face ESM state dict -> hooked transformer state dict
    """
    old_state_dict_keys = hf_esm_state_dict.keys()
    new_state_dict = {}

    old_to_new_weights = {
        "attention.self.query.weight":"attn.W_Q",
        "attention.self.key.weight":"attn.W_K",
        "attention.self.value.weight":"attn.W_V",
        "attention.output.dense.weight":"attn.W_O"
    }
    old_to_new_bias = {
        "attention.self.query.bias":"attn.b_Q",
        "attention.self.key.bias":"attn.b_K",
        "attention.self.value.bias":"attn.b_V",
        "attention.output.dense.bias":"attn.b_O"
    }
    
    for l in range(cfg.n_layers):
        l_keys = [x for x in old_state_dict_keys if f".{l}." in x]

        # weights
        for w in old_to_new_weights.keys():
            old_weight_prefix = f"esm.encoder.layer.{l}"
            new_weight_name = old_to_new_weights[w]
            new_state_dict[f"blocks.{l}.{new_weight_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_weight_prefix}.{w}"], "d_model (n_head d_head) -> n_head d_model d_head", n_head=cfg.n_heads)

            if l == 10:
                print(f"{old_weight_prefix}.{w} -> blocks.{l}.{new_weight_name}")
        
        #biases
        for b in old_to_new_bias.keys():
            old_weight_prefix = f"esm.encoder.layer.{l}"
            new_bias_name = old_to_new_bias[b]
            new_state_dict[f"blocks.{l}.{new_bias_name}"] = einops.rearrange(hf_esm_state_dict[f"{old_weight_prefix}.{b}"], "(n_head d_head) -> n_head d_head", n_head=cfg.n_heads)
            if l == 10:
                print(f"{old_weight_prefix}.{b} -> blocks.{l}.{new_bias_name}")
        
        if l == 10:
            print()

    return new_state_dict

In [107]:
test_fn = get_hooked_state_dict(model.state_dict(), hooked_esm_config)

esm.encoder.layer.10.attention.self.query.weight -> blocks.10.attn.W_Q
esm.encoder.layer.10.attention.self.key.weight -> blocks.10.attn.W_K
esm.encoder.layer.10.attention.self.value.weight -> blocks.10.attn.W_V
esm.encoder.layer.10.attention.output.dense.weight -> blocks.10.attn.W_O
esm.encoder.layer.10.attention.self.query.bias -> blocks.10.attn.b_Q
esm.encoder.layer.10.attention.self.key.bias -> blocks.10.attn.b_K
esm.encoder.layer.10.attention.self.value.bias -> blocks.10.attn.b_V
esm.encoder.layer.10.attention.output.dense.bias -> blocks.10.attn.b_O



In [100]:
print(test_fn["blocks.0.attn.b_Q"].shape)

torch.Size([20, 24])


In [67]:
print(test)

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
    

In [75]:
print("\n".join([x for x in test.state_dict().keys() if ".0." in x]))

blocks.0.ln1.w
blocks.0.ln1.b
blocks.0.ln2.w
blocks.0.ln2.b
blocks.0.attn.W_Q
blocks.0.attn.W_O
blocks.0.attn.b_Q
blocks.0.attn.b_O
blocks.0.attn.W_K
blocks.0.attn.W_V
blocks.0.attn.b_K
blocks.0.attn.b_V
blocks.0.attn.mask
blocks.0.attn.IGNORE
blocks.0.attn.rotary_sin
blocks.0.attn.rotary_cos
blocks.0.mlp.W_in
blocks.0.mlp.b_in
blocks.0.mlp.W_out
blocks.0.mlp.b_out


In [101]:
print(test.state_dict()["blocks.0.attn.W_O"].shape)

torch.Size([20, 24, 480])


In [21]:
print("\n".join([x for x in model.state_dict().keys() if ".0." in x]))

esm.encoder.layer.0.attention.self.query.weight
esm.encoder.layer.0.attention.self.query.bias
esm.encoder.layer.0.attention.self.key.weight
esm.encoder.layer.0.attention.self.key.bias
esm.encoder.layer.0.attention.self.value.weight
esm.encoder.layer.0.attention.self.value.bias
esm.encoder.layer.0.attention.self.rotary_embeddings.inv_freq
esm.encoder.layer.0.attention.output.dense.weight
esm.encoder.layer.0.attention.output.dense.bias
esm.encoder.layer.0.attention.LayerNorm.weight
esm.encoder.layer.0.attention.LayerNorm.bias
esm.encoder.layer.0.intermediate.dense.weight
esm.encoder.layer.0.intermediate.dense.bias
esm.encoder.layer.0.output.dense.weight
esm.encoder.layer.0.output.dense.bias
esm.encoder.layer.0.LayerNorm.weight
esm.encoder.layer.0.LayerNorm.bias


In [97]:
print(model.state_dict()["esm.encoder.layer.0.attention.self.query.bias"].shape)

torch.Size([480])


In [73]:
print(model)

EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 480, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-11): 12 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=480, out_features=480, bias=True)
              (key): Linear(in_features=480, out_features=480, bias=True)
              (value): Linear(in_features=480, out_features=480, bias=True)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=480, out_features=480, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((480,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=480, out_fe

In [ ]:
test = HookedTransformer.from_pretrained("

In [11]:
# load model and tokenizer
HF_model_name = "hugohrban/progen2-small"

model = AutoModelForCausalLM.from_pretrained(HF_model_name, trust_remote_code=True, torch_dtype="auto")
tokenizer = Tokenizer.from_pretrained(HF_model_name)
tokenizer.no_padding()

# prepare input
prompt = "1MEVVIVTGMSGAGK"
input_ids = torch.tensor(tokenizer.encode(prompt).ids).to(model.device)

# forward pass
logits = model(input_ids).logits

# print output probabilities
next_token_logits = logits[-1, :]
next_token_probs = F.softmax(next_token_logits, dim=-1)
for i in range(tokenizer.get_vocab_size(with_added_tokens=False)):
    print(f"{tokenizer.id_to_token(i)}: {100 * next_token_probs[i].item():.2f} %")

<|pad|>: 0.00 %
<|bos|>: 0.00 %
<|eos|>: 0.00 %
1: 0.00 %
2: 0.00 %
A: 0.45 %
B: 0.00 %
C: 0.06 %
D: 0.10 %
E: 0.04 %
F: 0.04 %
G: 0.68 %
H: 0.09 %
I: 0.11 %
K: 0.11 %
L: 0.07 %
M: 0.07 %
N: 0.25 %
O: 0.00 %
P: 0.01 %
Q: 0.07 %
R: 0.11 %
S: 63.64 %
T: 33.91 %
U: 0.00 %
V: 0.12 %
W: 0.01 %
X: 0.01 %
Y: 0.03 %
Z: 0.00 %
